# Руководство по генеалогическому древу в UnifyWeaver

Этот интерактивный блокнот демонстрирует, как использовать UnifyWeaver для компиляции предикатов Prolog в скрипты Bash.

## Предварительные требования

- Установлен SWI-Prolog
- Доступна библиотека UnifyWeaver
- Установлено ядро Prolog для Jupyter (`pip install prolog-jupyter-kernel`)

## Цели обучения

По завершении работы с этим блокнотом вы сможете:
1. Определять факты и правила на Prolog
2. Использовать UnifyWeaver для компиляции предикатов в Bash
3. Тестировать сгенерированные скрипты Bash
4. Понимать принципы компиляции транзитивного замыкания

## Шаг 1: Инициализация окружения UnifyWeaver

Сначала необходимо загрузить модули UnifyWeaver. Мы используем файл `init.pl` из каталога education.

In [ ]:
% Загрузить файл инициализации
['../init'].

## Шаг 2: Определение родственных связей

Определим несколько отношений между родителями и детьми на основе библейского генеалогического древа.

In [ ]:
% Определить факты parent
:- dynamic parent/2.

parent(abraham, isaac).
parent(abraham, ishmael).
parent(isaac, jacob).
parent(isaac, esau).
parent(jacob, reuben).
parent(jacob, simeon).
parent(jacob, levi).
parent(jacob, judah).

## Шаг 3: Тестирование запросов о родителях

Перед компиляцией проверим корректность данных с помощью нескольких запросов на Prolog.

In [ ]:
% Запрос: Кто дети Авраама?
parent(abraham, Child).

In [ ]:
% Запрос: Кто дети Иакова?
parent(jacob, Child).

## Шаг 4: Определение отношения предка

Теперь определим транзитивное замыкание — отношение `ancestor`.

In [ ]:
% Определить ancestor как транзитивное замыкание parent
:- dynamic ancestor/2.

% Базовый случай: родитель является предком
ancestor(X, Y) :- parent(X, Y).

% Рекурсивный случай: если X — родитель Y, а Y — предок Z, то X — предок Z
ancestor(X, Z) :- parent(X, Y), ancestor(Y, Z).

## Шаг 5: Тестирование запросов о предках

Проверим правильность работы нашего предиката предка.

In [ ]:
% Запрос: Является ли Авраам предком Иакова?
( ancestor(abraham, jacob) ->
    writeln('Yes: Abraham is an ancestor of Jacob')
;
    writeln('No: Abraham is not an ancestor of Jacob')
).

In [ ]:
% Запрос: Кто все потомки Авраама?
ancestor(abraham, Descendant).

## Шаг 6: Компиляция Parent в Bash

Переходим к самому интересному — скомпилируем наши факты `parent/2` в скрипт Bash!

In [ ]:
% Загрузить потоковый компилятор
\+ \+ (
    use_module(unifyweaver(core/stream_compiler)),

    % Скомпилировать факты parent в bash
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    writeln('Generated Bash code for parent/2:'),
    writeln(_BashCode)
).

## Шаг 7: Сохранение скрипта Parent

Сохраним сгенерированный код Bash в файл.

In [ ]:
% Сохранить в файл
\+ \+ (
    stream_compiler:compile_facts(parent, 2, [], _BashCode),
    setup_call_cleanup(
        open('../output/parent.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/parent.sh')
).

## Шаг 8: Компиляция Ancestor в Bash

Теперь скомпилируем предикат `ancestor/2`, использующий рекурсию.

In [ ]:
% Загрузить рекурсивный компилятор
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),

    % Скомпилировать ancestor в bash
    compile_recursive(ancestor/2, [], _BashCode),
    writeln('Generated Bash code for ancestor/2:'),
    writeln(_BashCode)
).

## Шаг 9: Сохранение скрипта Ancestor

Сохраните скрипт предка в файл.

In [ ]:
% Сохранить в файл
\+ \+ (
    compile_recursive(ancestor/2, [], _BashCode),
    setup_call_cleanup(
        open('../output/ancestor.sh', write, _Stream),
        write(_Stream, _BashCode),
        close(_Stream)),
    writeln('Saved to ../output/ancestor.sh')
).

## Шаг 10: Тестирование сгенерированных скриптов

Протестируем сгенерированные скрипты Bash! Для выполнения команд bash мы воспользуемся магической командой `%%bash`.

In [ ]:
%%bash
# Подключить скрипт parent с помощью source
source ../output/parent.sh

# Тест: Кто дети Авраама?
echo "Дети Авраама:"
parent abraham

In [ ]:
%%bash
# Подключить оба скрипта с помощью source
source ../output/parent.sh
source ../output/ancestor.sh

# Тест: Кто потомки Авраама?
echo "Потомки Авраама:"
ancestor abraham

In [ ]:
%%bash
# Подключить оба скрипта с помощью source
source ../output/parent.sh
source ../output/ancestor.sh

# Тест: Является ли Авраам предком Иуды?
if ancestor abraham judah >/dev/null 2>&1; then
    echo "✓ Да, Авраам — предок Иуды"
else
    echo "✗ Нет"
fi

## Шаг 11: Анализ стратегии компиляции

Разберем, что выполнил UnifyWeaver:

1. **Компиляция parent**: использован `stream_compiler` для создания простой потоковой функции, выводящей все пары «родитель-потомок»

2. **Компиляция ancestor**: обнаружен паттерн транзитивного замыкания и применена оптимизация BFS (поиск в ширину) для эффективного вычисления всех достижимых предков

Проверим примененную стратегию компиляции:

In [ ]:
% Проверить, классифицирован ли ancestor как рекурсивный
\+ \+ (
    use_module(unifyweaver(core/recursive_compiler)),
    recursive_compiler:classify_predicate(ancestor/2, _Classification),
    format('Ancestor classification: ~w~n', [_Classification])
).

## Резюме

В этом блокноте вы узнали:

✅ Как определять факты и правила на Prolog

✅ Как использовать `stream_compiler` в UnifyWeaver для фактов

✅ Как использовать `recursive_compiler` в UnifyWeaver для рекурсивных предикатов

✅ Как тестировать сгенерированные скрипты Bash

✅ Что UnifyWeaver автоматически обнаруживает транзитивное замыкание и применяет оптимизацию BFS

## Следующие шаги

Попробуйте выполнить следующие упражнения:

1. Добавить больше членов семьи в генеалогическое древо
2. Определить предикат `grandparent/2` и скомпилировать его
3. Создать предикат `sibling/2` (два человека с общими родителями)
4. Изучить сгенерированный код Bash, чтобы детально понять алгоритм BFS

Переходите к **Блокноту 2: Сравнение паттернов рекурсии**, чтобы изучить продвинутые паттерны рекурсии!